# 🚍🚆🚊 Transit Equity in Montreal
## Do we serve everyone equally? A spatial analysis of STM service *vs.* neighbourhood demographics.

**Author:** Bruna Bado  
**Last updated:** June 2026  
**Data sources:** STM GTFS Static Feed & Statistics Canada 2021 Census

---

### Abstract

Transit agencies are often evaluated on punctuality. But a harder and more important question is whether service runs where it matters most.

This analysis examines transit equity in Montreal by measuring the relationship between STM transit service levels and the socioeconomic characteristics of each neighbourhood.

Using a spatial analysis of GTFS schedule data overlaid onto 2021 Census tract demographics, we identify which parts of the Montreal metropolitan region have the weakest service, and ask whether those gaps fall disproportionately on lower-income residents and visible minority communities.

**Research questions:**  
**1.** Which Montreal Census Tracts have the least transit service, measured by stop density and weekly trip frequency?  
**2.** Are low-service areas correlated with lower median income, higher proportions of visible minorities, or higher rates of low income?  
**3.** Which patterns emerge that could inform transit equity policy?   

 
---

## 1. Setup

In [37]:
# Standard library ──────────────────────────────────────────────────────────
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Network ────────────────────────────────────────────────────────────────────
import requests

# Data manipulation ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# Geospatial ─────────────────────────────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import Point

# Visualization ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from folium.plugins import HeatMap
from branca.colormap import linear

# Statistics ─────────────────────────────────────────────────────────────────
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Display settings ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_theme(style='whitegrid', palette='muted')

# File paths (relative to the notebooks/ directory) ─────────────────────────
DATA_RAW       = os.path.join('..', 'data', 'raw')
DATA_PROCESSED = os.path.join('..', 'data', 'processed')
FIGURES        = os.path.join('..', 'figures')

for path in [DATA_RAW, DATA_PROCESSED, FIGURES,
             os.path.join(DATA_RAW, 'gtfs_stm'),
             os.path.join(DATA_RAW, 'census')]:
    os.makedirs(path, exist_ok=True)

print("✅ All libraries loaded successfully.")
print(f"   Working directory: {os.getcwd()}")

✅ All libraries loaded successfully.
   Working directory: c:\Users\bruna\Documents\Data Analysis\transit-equity-montreal\notebooks


## 2. Data Sources

This analysis relies on 3 publicly available datasets:

|   | Source | Description | Format | Geography |
|---|--------|-------------|--------|-----------|
| 1 | **STM GTFS** | Scheduled routes, stops, and trip frequencies for all STM buses and metro lines | ZIP of CSVs | Stop-level |
| 2 | **StatsCan 2021 Census — Boundaries** | Geographic boundaries for Census Tracts in the Montreal CMA | Shapefile | Census Tract |
| 3 | **StatsCan 2021 Census — Profile** | Demographic, income, and socioeconomic indicators | CSV | Census Tract |

### What is GTFS?
GTFS (General Transit Feed Specification) is an open standard originally developed by Google and now adopted worldwide. A GTFS feed is a ZIP archive containing several inter-related CSV tables: routes, trips, stops, stop times, and calendar data. It is the same format that apps like Transit use to show riders when the next bus arrives. This analysis uses the **static** GTFS feed, which represents the published scheduled timetable.

### What is a Census Tract?
Census Tracts (CTs) are small, stable geographic units designed by Statistics Canada to have populations of roughly 2,500-8,000 people. They are the finest-grained geography at which StatsCan publishes full demographic and income breakdowns, making them the ideal unit for neighbourhood-level equity analysis.

### Data Currency
- The GTFS feed represents current scheduled service.  
- The census data is from 2021. Demographic shifts may have occurred since then, but the 2021 long-form census remains the most granular, reliable, and publicly available source for the socioeconomic indicators used here.

## 3. Data Loading and Preprocessing

### 3.1. GTFS - STM Montreal

In [38]:
# Reusable download helper ───────────────────────────────────────────────────

     # It downloads a file from `url` to `dest_path`. It skips if the file already exists, so it's safe to re-run.

def download_file(url: str, dest_path: str, label: str = "") -> bool:

    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / 1_048_576
        print(f"✅ {label} already exists ({size_mb:.1f} MB)... Skipping file.")
        return True

    print(f"⬇️  Downloading {label} ...")
    try:
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65_536):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r   {downloaded / total * 100:5.1f}%  "
                          f"({downloaded/1_048_576:.1f} / {total/1_048_576:.1f} MB)",
                          end="")
        print(f"\n✅ Saved to: {dest_path}")
        return True
    except Exception as e:
        print(f"\n❌ Download failed: {e}")
        if os.path.exists(dest_path):
            os.remove(dest_path)   # remove partial download
        return False

In [39]:
# Download STM GTFS feed ─────────────────────────────────────────────────────

GTFS_URL  = "https://www.stm.info/sites/default/files/gtfs/gtfs_stm.zip"
GTFS_PATH = os.path.join(DATA_RAW, 'gtfs_stm', 'gtfs_stm.zip')

success = download_file(GTFS_URL, GTFS_PATH, label="STM GTFS feed")

if not success:
    print("""
──────────────────────────────────────────────────────────────────
⚠️  Automatic download failed. Manual steps:
    1. Go to:  https://www.stm.info/en/about/developers
    2. Download the GTFS static feed (.zip file)
    3. Save it as:  data/raw/gtfs_stm/gtfs_stm.zip
    4. Re-run this cell.
──────────────────────────────────────────────────────────────────
""")

✅ STM GTFS feed already exists (57.1 MB)... Skipping file.


In [40]:
# Parse GTFS tables from the ZIP archive ─────────────────────────────────────

     # We can read each table directly from memory, so there's no need to extract to disk.

print("📂 Parsing GTFS tables...\n")

with zipfile.ZipFile(GTFS_PATH, 'r') as z:
    print(f"Files inside ZIP: {z.namelist()}\n")

    stops      = pd.read_csv(z.open('stops.txt'))
    routes     = pd.read_csv(z.open('routes.txt'))
    trips      = pd.read_csv(z.open('trips.txt'))
    stop_times = pd.read_csv(z.open('stop_times.txt'),
                              dtype={'arrival_time': str, 'departure_time': str})
    calendar   = pd.read_csv(z.open('calendar.txt'))

for name, df in [('stops', stops), ('routes', routes), ('trips', trips),
                  ('stop_times', stop_times), ('calendar', calendar)]:
    print(f"   {name:<12} {len(df):>9,} rows  ×  {df.shape[1]} columns")

print("\n✅ All GTFS tables have been loaded.")

📂 Parsing GTFS tables...

Files inside ZIP: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'directions.txt', 'feed_info.txt', 'route_patterns.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'translations.txt', 'trips.txt']

   stops            9,188 rows  ×  9 columns
   routes             231 rows  ×  10 columns
   trips          203,056 rows  ×  8 columns
   stop_times   7,151,705 rows  ×  6 columns
   calendar           132 rows  ×  10 columns

✅ All GTFS tables have been loaded.


In [41]:
# Inspect the structure of each GTFS table ───────────────────────────────────

for name, df in [('STOPS', stops), ('ROUTES', routes),
                  ('TRIPS', trips), ('STOP_TIMES', stop_times),
                  ('CALENDAR', calendar)]:
    print(f"\n{name} {'─'*(52-len(name))}")
    display(df.head(3))
    print(f"dtypes: {dict(df.dtypes)}")


STOPS ───────────────────────────────────────────────


,stop_id,stop_code,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding
0,STATION_M118,10118,STATION ANGRIGNON,45.45,-73.60,NaN,1,NaN,1
1,43,10118,Station Angrignon,45.45,-73.60,https://www.stm.info/fr/infos/reseaux/metro/an...,0,STATION_M118,1
2,43-01,10118,Station Angrignon,45.45,-73.60,NaN,2,STATION_M118,1


dtypes: {'stop_id': <StringDtype(storage='python', na_value=nan)>, 'stop_code': dtype('int64'), 'stop_name': <StringDtype(storage='python', na_value=nan)>, 'stop_lat': dtype('float64'), 'stop_lon': dtype('float64'), 'stop_url': <StringDtype(storage='python', na_value=nan)>, 'location_type': dtype('int64'), 'parent_station': <StringDtype(storage='python', na_value=nan)>, 'wheelchair_boarding': dtype('int64')}

ROUTES ──────────────────────────────────────────────


,route_id,agency_id,route_short_name,route_long_name,route_type,route_url,route_color,route_text_color,route_desc,route_desc_detail
0,1,STM,1,Ligne 1 - Verte,1,https://www.stm.info/fr/infos/reseaux/metro/li...,00B300,FFFFFF,NaN,NaN
1,2,STM,2,Ligne 2 - Orange,1,https://www.stm.info/fr/infos/reseaux/metro/li...,D95700,FFFFFF,NaN,NaN
2,4,STM,4,Ligne 4 - Jaune,1,https://www.stm.info/fr/infos/reseaux/metro/li...,FFD900,000000,NaN,NaN


dtypes: {'route_id': dtype('int64'), 'agency_id': <StringDtype(storage='python', na_value=nan)>, 'route_short_name': dtype('int64'), 'route_long_name': <StringDtype(storage='python', na_value=nan)>, 'route_type': dtype('int64'), 'route_url': <StringDtype(storage='python', na_value=nan)>, 'route_color': <StringDtype(storage='python', na_value=nan)>, 'route_text_color': <StringDtype(storage='python', na_value=nan)>, 'route_desc': <StringDtype(storage='python', na_value=nan)>, 'route_desc_detail': <StringDtype(storage='python', na_value=nan)>}

TRIPS ───────────────────────────────────────────────


...

dtypes: {'route_id': dtype('int64'), 'service_id': <StringDtype(storage='python', na_value=nan)>, 'trip_id': dtype('int64'), 'trip_headsign': <StringDtype(storage='python', na_value=nan)>, 'direction_id': dtype('int64'), 'shape_id': <StringDtype(storage='python', na_value=nan)>, 'wheelchair_accessible': dtype('int64'), 'route_pattern_id': <StringDtype(storage='python', na_value=nan)>}

STOP_TIMES ──────────────────────────────────────────


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type
0,295375000,05:58:00,05:58:00,55545,1,0
1,295375000,05:58:15,05:58:15,55568,2,0
2,295375000,05:59:07,05:59:07,55584,3,0


dtypes: {'trip_id': dtype('int64'), 'arrival_time': <StringDtype(storage='python', na_value=nan)>, 'departure_time': <StringDtype(storage='python', na_value=nan)>, 'stop_id': dtype('int64'), 'stop_sequence': dtype('int64'), 'pickup_type': dtype('int64')}

CALENDAR ────────────────────────────────────────────


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,26M-GLOBAUX-02-S,1,1,1,1,1,0,0,20260323,20260612
1,26M-H50M000S-70-S,1,1,1,1,1,0,0,20260323,20260515
2,26M-H50M000S-70-SV,0,0,0,0,1,0,0,20260323,20260515


dtypes: {'service_id': <StringDtype(storage='python', na_value=nan)>, 'monday': dtype('int64'), 'tuesday': dtype('int64'), 'wednesday': dtype('int64'), 'thursday': dtype('int64'), 'friday': dtype('int64'), 'saturday': dtype('int64'), 'sunday': dtype('int64'), 'start_date': dtype('int64'), 'end_date': dtype('int64')}


### 3.2. Census Tract Boundaries

In [42]:
# Download Census Tract boundary shapefile (Canada, 2021) ───────────────────

     # The "lct" file contains Census Tract polygons for all of Canada. These are the digital boundary files for all standard geographies, published by Statistics Canada.
     # Source: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/

CT_SHAPE_URL  = ("https://www12.statcan.gc.ca/census-recensement/2021/geo/"
                 "sip-pis/boundary-limites/files-fichiers/lct_000b21a_e.zip")
CT_SHAPE_PATH = os.path.join(DATA_RAW, 'census', 'ct_boundaries.zip')

download_file(CT_SHAPE_URL, CT_SHAPE_PATH, label="Census Tract boundaries (StatsCan)")

✅ Census Tract boundaries (StatsCan) already exists (12.8 MB)... Skipping file.


True

In [43]:
# Load Census Tract shapefile into memory ─────────────────────────────────────

print("📦 Loading Census Tract shapefile (Canada-wide)...")

ct_canada = gpd.read_file(f"zip://{CT_SHAPE_PATH}")

print(f"   Full Canada dataset : {len(ct_canada):,} census tracts")
print(f"   CRS                 : {ct_canada.crs}")
print(f"   Columns             : {list(ct_canada.columns)}")

📦 Loading Census Tract shapefile (Canada-wide)...
   Full Canada dataset : 6,247 census tracts
   CRS                 : EPSG:3347
   Columns             : ['CTUID', 'DGUID', 'CTNAME', 'LANDAREA', 'PRUID', 'geometry']


In [44]:
# Verify Quebec CTUIDs before filtering ─────────────────────────────────────

qc = ct_canada[ct_canada['PRUID'] == '24']
print(f"Total Quebec CTs : {len(qc)}")
print(f"\nSample CTUIDs:")
print(qc['CTUID'].head(15).tolist())
print(f"\nCTUID starts with '462' (Montreal CMA):")
print(qc[qc['CTUID'].str.startswith('462')]['CTUID'].head(10).tolist())

Total Quebec CTs : 1480

Sample CTUIDs:
['4210002.00', '4210003.00', '4210004.00', '4210005.00', '4210006.00', '4210007.00', '4210008.00', '4210009.00', '4210010.00', '4210011.00', '4210012.00', '4210013.00', '4210014.00', '4210015.00', '4210016.00']

CTUID starts with '462' (Montreal CMA):
['4620725.07', '4620652.04', '4620887.05', '4620887.06', '4620887.07', '4620688.03', '4620688.04', '4620001.00', '4620002.00', '4620003.00']


In [45]:
# Filter to Montreal CMA ─────────────────────────────────────────────────────

     # The CTUID format in this file is CCC (3-digit CMA code) + TTTT.TT (CT number).
     # The province is not embedded in CTUID. It's stored separately in PRUID.

     # Montreal CMA code is '462' - that means that all Montreal CTUIDs start with '462'.
     # Since CMA codes are nationally unique, no other province has a CMA starting with 462.
     # However, requiring PRUID == '24' (Quebec) as a safety belt is good practice.

montreal_ct = ct_canada[
    (ct_canada['PRUID'] == '24') &
    (ct_canada['CTUID'].str.startswith('462'))
].copy()

montreal_ct = montreal_ct.to_crs(epsg=4326)

print(f"Montreal CMA census tracts : {len(montreal_ct):,}")
print(f"CRS after reprojection     : {montreal_ct.crs}")
print(f"\nColumns : {list(montreal_ct.columns)}")
display(montreal_ct.head(3))

Montreal CMA census tracts : 1,004
CRS after reprojection     : EPSG:4326

Columns : ['CTUID', 'DGUID', 'CTNAME', 'LANDAREA', 'PRUID', 'geometry']


,CTUID,DGUID,CTNAME,LANDAREA,PRUID,geometry
161,4620725.07,2021S05074620725.07,0725.07,1.47,24,"POLYGON ((-73.93606 45.58568, -73.93672 45.584..."
162,4620652.04,2021S05074620652.04,0652.04,1.27,24,"POLYGON ((-73.80088 45.53048, -73.80078 45.530..."
197,4620887.05,2021S05074620887.05,0887.05,1.98,24,"POLYGON ((-73.43923 45.57119, -73.43982 45.571..."


### 3.3. Census Demographic Data

The demographic variables come from the **2021 Census Profile** for Census Metropolitan Areas and Census Tracts, published by Statistics Canada.

**Download steps:**
1. Go to: https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger.cfm.
2. Find the row *"Census metropolitan areas, tracted census agglomerations and census tracts"* → click **CSV**.
3. Save the resulting `.zip` into `data/raw/census/`.
4. Rename it to exactly: `census_profile_ct.zip`.
5. Run the cells below — no manual extraction needed.

Since the file is large, we used chunked reading and filter to Montreal Census Tracts during loading to keep the memory use more manageable.

In [46]:
# Load census data (chunked read, filter to Montreal CTs) ────────────────────

     # Key facts about this file:
     #   - Format       : long (one row per characteristic per geography)
     #   - GEO_LEVEL    : 'Census metropolitan area' or 'Census tract'
     #   - DGUID format : '2021S0507' + CTUID for Census Tracts
     #   - Montreal CTs : DGUID starts with '2021S0507462' (CMA code 462)

CENSUS_ZIP          = os.path.join(DATA_RAW, 'census', 'census_profile_ct.zip')
CSV_NAME            = '98-401-X2021007_English_CSV_data.csv'
MONTREAL_CT_PREFIX  = '2021S0507462'   # DGUID prefix for all Montreal CTs
CHUNK_SIZE          = 100_000

USEFUL_COLS = ['DGUID', 'ALT_GEO_CODE', 'GEO_LEVEL', 'GEO_NAME',
               'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']

chunks = []

print("📊 Reading census file in chunks, filtering to Montreal CTs...")

with zipfile.ZipFile(CENSUS_ZIP, 'r') as z:
    with z.open(CSV_NAME) as f:
        reader = pd.read_csv(
            f,
            usecols=USEFUL_COLS,
            encoding='latin-1',
            low_memory=False,
            chunksize=CHUNK_SIZE
        )
        for i, chunk in enumerate(reader):
            montreal_rows = chunk[
                chunk['DGUID'].str.startswith(MONTREAL_CT_PREFIX, na=False)
            ].copy()

            # Because StatsCan uses indentation to show hierarchy, we need to strip leading/trailing whitespace from characteristic names. Otherwise, it would break exact matching.
            montreal_rows['CHARACTERISTIC_NAME'] = (
                montreal_rows['CHARACTERISTIC_NAME'].str.strip()
            )
            if len(montreal_rows) > 0:
                chunks.append(montreal_rows)
            if i % 5 == 0:
                print(f"\r    Scanned ~{(i + 1) * CHUNK_SIZE:,} rows...", end="")

census_mtl = pd.concat(chunks, ignore_index=True)

print(f"\n\n✅ Done!")
print(f"    Montreal CT rows loaded : {len(census_mtl):,}")
print(f"    Unique Census Tracts    : {census_mtl['DGUID'].nunique()}")
print(f"    Unique characteristics  : {census_mtl['CHARACTERISTIC_NAME'].nunique()}")
print(f"\nSample rows:")
display(census_mtl.head(5))

📊 Reading census file in chunks, filtering to Montreal CTs...
    Scanned ~16,600,000 rows...

✅ Done!
    Montreal CT rows loaded : 2,641,524
    Unique Census Tracts    : 1004
    Unique characteristics  : 1301

Sample rows:


,DGUID,ALT_GEO_CODE,GEO_LEVEL,GEO_NAME,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
0,2021S05074620001.00,4620001.00,Census tract,4620001.00,"Population, 2021",2684.00
1,2021S05074620001.00,4620001.00,Census tract,4620001.00,"Population, 2016",2638.00
2,2021S05074620001.00,4620001.00,Census tract,4620001.00,"Population percentage change, 2016 to 2021",1.70
3,2021S05074620001.00,4620001.00,Census tract,4620001.00,Total private dwellings,1463.00
4,2021S05074620001.00,4620001.00,Census tract,4620001.00,Private dwellings occupied by usual residents,1402.00


In [47]:
# Find the exact characteristic names we need ───────────────────────────────

     # Because characteristic names sometimes differ slightly between census releases, we need to search for keywords and print all matches, so we can use the exact strings below.

all_chars = census_mtl['CHARACTERISTIC_NAME'].unique()
print(f"Total unique characteristics: {len(all_chars)}\n")

search_terms = [
    'population, 2021',
    'density',
    'median total income',
    'low income',
    'visible minority',
    'renter',
    'tenure',
]

for term in search_terms:
    matches = [c for c in all_chars if term.lower() in c.lower()]
    print(f"── '{term}'  ({len(matches)} match{'es' if len(matches) != 1 else ''}) ─────")
    for m in matches[:6]:
        print(f"   {m}")
    print()

Total unique characteristics: 1301

── 'population, 2021'  (1 match) ─────
   Population, 2021

── 'density'  (1 match) ─────
   Population density per square kilometre

── 'median total income'  (10 matches) ─────
   Median total income in 2020 among recipients ($)
   Median total income in 2019 among recipients ($)
   Median total income of household in 2020 ($)
   Median total income of one-person households in 2020 ($)
   Median total income of two-or-more-person households in 2020 ($)
   Median total income of economic family in 2020 ($)

── 'low income'  (4 matches) ─────
   In low income based on the Low-income measure, after tax (LIM-AT)
   Prevalence of low income based on the Low-income measure, after tax (LIM-AT) (%)
   In low income based on the Low-income cut-offs, after tax (LICO-AT)
   Prevalence of low income based on the Low-income cut-offs, after tax (LICO-AT) (%)

── 'visible minority'  (4 matches) ─────
   Total - Visible minority for the population in private house

In [48]:
# Extract our target variables and pivot to wide format ─────────────────────

     # If the diagnostic cell above shows different names in our file, the strings on the LEFT side of each pair below need to be updated.
     # (The strings on the RIGHT are the internal column names, so they shouldn't be changed.)

CHARACTERISTICS = {
    'Population, 2021'
        : 'population',
    'Population density per square kilometre'
        : 'pop_density',
    'Median total income of household in 2020 ($)'
        : 'median_hh_income',
    'Prevalence of low income based on the Low-income measure, after tax (LIM-AT) (%)'
        : 'pct_low_income',
    'Total - Visible minority for the population in private households - 25% sample data'
        : 'total_vis_min_base',
    'Total visible minority population'
        : 'vis_min_count',
    'Total - Private households by tenure - 25% sample data'
        : 'total_households',
    'Renter'
        : 'renter_count',
}

# Verify all characteristics were actually found before pivoting ─────────────

found     = set(census_mtl['CHARACTERISTIC_NAME'].unique())
not_found = [k for k in CHARACTERISTICS if k not in found]

if not_found:
    print("⚠️  These characteristics were NOT found. Please update the strings above:")
    for nf in not_found:
        print(f"    '{nf}'")
    print("\nRe-run the diagnostic cell to find the correct names.")
else:
    print("✅ All characteristics found. Pivoting...\n")

    census_wide = (
        census_mtl[census_mtl['CHARACTERISTIC_NAME'].isin(CHARACTERISTICS.keys())]
        .pivot_table(
            index='DGUID',
            columns='CHARACTERISTIC_NAME',
            values='C1_COUNT_TOTAL',
            aggfunc='first'
        )
        .rename(columns=CHARACTERISTICS)
        .reset_index()
    )

    # Derived columns
    census_wide['pct_visible_minority'] = (
        census_wide['vis_min_count'] / census_wide['total_vis_min_base'] * 100
    )
    census_wide['pct_renter'] = (
        census_wide['renter_count'] / census_wide['total_households'] * 100
    )

    print(f"Shape: {census_wide.shape[0]} CTs × {census_wide.shape[1]} columns")
    display(census_wide.head())

✅ All characteristics found. Pivoting...

Shape: 1002 CTs × 11 columns


CHARACTERISTIC_NAME,DGUID,median_hh_income,pop_density,population,pct_low_income,renter_count,total_households,total_vis_min_base,vis_min_count,pct_visible_minority,pct_renter
0,2021S05074620001.00,63600.00,5780.70,2684.00,14.50,770.00,1380.00,2570.00,595.00,23.15,55.80
1,2021S05074620002.00,57600.00,9797.80,3780.00,17.40,1200.00,1950.00,3790.00,865.00,22.82,61.54
2,2021S05074620003.00,62000.00,8917.70,6600.00,15.80,1835.00,3030.00,6640.00,1840.00,27.71,60.56
3,2021S05074620004.00,60000.00,7505.60,3355.00,14.00,1000.00,1630.00,3160.00,710.00,22.47,61.35
4,2021S05074620005.00,58800.00,5617.50,3175.00,16.10,1125.00,1655.00,3180.00,430.00,13.52,67.98


In [49]:
# Merge census data onto geographic boundaries ──────────────────────────────

     # Both sides use DGUID as the identifier, so no key manipulation is needed.
     # Left join keeps all 1,004 Montreal CTs, while the the unmatched ones get NaN demographics.

master = montreal_ct.merge(census_wide, on='DGUID', how='left')

# Merge quality check ────────────────────────────────────────────────────────

total          = len(master)
matched        = master['population'].notna().sum()
unmatched      = total - matched

print(f"Total CTs in shapefile   : {total:,}")
print(f"Matched to census data   : {matched:,}")
print(f"Unmatched (no census data): {unmatched:,}")

if unmatched > 0:
    print(f"\n⚠️ Unmatched CTs (first 5):")
    display(master[master['population'].isna()][['CTUID', 'DGUID', 'CTNAME']].head())
    print("Note: A small number of unmatched CTs is normal, as some CTs may have suppressed data due to small populations.")

print(f"\nMissing value counts per column:")
demo_cols = list(CHARACTERISTICS.values()) + ['pct_visible_minority', 'pct_renter']
display(master[demo_cols].isnull().sum().rename("missing"))

# Save to disk ───────────────────────────────────────────────────────────────

out_path = os.path.join(DATA_PROCESSED, 'montreal_ct_master.gpkg')
master.to_file(out_path, driver='GPKG')
print(f"\n✅ Master dataset saved to: {out_path}")
print(f"   Shape: {master.shape}")

Total CTs in shapefile   : 1,004
Matched to census data   : 1,002
Unmatched (no census data): 2

⚠️ Unmatched CTs (first 5):


,CTUID,DGUID,CTNAME
387,4620832.00,2021S05074620832.00,0832.00
642,4620732.03,2021S05074620732.03,0732.03


Note: A small number of unmatched CTs is normal, as some CTs may have suppressed data due to small populations.

Missing value counts per column:


population               2
pop_density              2
median_hh_income        18
pct_low_income          18
total_vis_min_base      18
vis_min_count           18
total_households        18
renter_count            18
pct_visible_minority    18
pct_renter              18
Name: missing, dtype: int64


✅ Master dataset saved to: ..\data\processed\montreal_ct_master.gpkg
   Shape: (1004, 16)
